### Download and load the data

In [11]:
import kagglehub
import pandas as pd
import os

# Download latest version
path = kagglehub.dataset_download("yasserh/titanic-dataset")

print("Path to dataset files:", path)
dataset = pd.read_csv(os.path.join(path, "Titanic-Dataset.csv"))  # swap in the real filename from the listing

Path to dataset files: /Users/mattwilliams/.cache/kagglehub/datasets/yasserh/titanic-dataset/versions/1


### Train model

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

features = ['Pclass', 'Sex', 'SibSp', 'Parch']
X = pd.get_dummies(dataset[features])
y = dataset['Survived']


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

### Dummy model for comparison

In [13]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

y_pred_base = baseline.predict(X_test)
y_proba_base = baseline.predict_proba(X_test)[:, 1]

evaluate(baseline, X_test, y_test)

{'accuracy': 0.6145251396648045,
 'roc_auc': 0.5,
 'recall_class_0': 1.0,
 'recall_class_1': 0.0}

### Evaluate model

In [ ]:

from sklearn.metrics import accuracy_score, recall_score, roc_auc_score


def evaluate(model, X_test, y_test) -> dict:
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    recalls = recall_score(y_test, y_pred, average=None)  # one value per class

    return {
        "accuracy": float(accuracy_score(y_test, y_pred)),
        "roc_auc": float(roc_auc_score(y_test, y_proba)),
        "recall_class_0": float(recalls[0]),
        "recall_class_1": float(recalls[1]),
    }

In [15]:
print("model:   ", evaluate(model, X_test, y_test))
print("baseline:", evaluate(baseline, X_test, y_test))

model:    {'accuracy': 0.770949720670391, 'roc_auc': 0.8380764163372859, 'recall_class_0': 0.8909090909090909, 'recall_class_1': 0.5797101449275363}
baseline: {'accuracy': 0.6145251396648045, 'roc_auc': 0.5, 'recall_class_0': 1.0, 'recall_class_1': 0.0}
